# Build the real-text annotation set

Downloads **HumAID** — 76,484 human-labelled tweets from 19 real disasters
— samples a balanced subset, and writes two CSVs for two people to label
independently.

Run with **Runtime → Run all**, then download the two CSVs at the end.

**Why this exists:** every number in the project so far comes from text the
authors or a generator wrote. That is the one thing a reviewer will not
accept. This replaces it with text real people wrote during real disasters.

**What HumAID does not give you:** its labels say what a tweet is *about*
(information type), not what the emergency *is*. There is no label for
casualty count, mobility, hazard or severity, and no class for TRAPPED,
FLOOD or FIRE. So HumAID supplies the text; the labels are still yours to
make. It is also **English only** — Hindi has to be collected separately.

License: CC BY-NC-SA 4.0. Cite the ICWSM 2021 paper and keep it
non-commercial.


In [ ]:
!pip install -q datasets
from datasets import load_dataset
import pandas as pd, numpy as np

# verification_mode: the dataset card declares a split called 'dev' while
# the published files produce 'validation', and the default check treats
# that mismatch as a fatal error. The data itself is complete.
ds = load_dataset('QCRI/HumAID-all', verification_mode='no_checks')
df = pd.concat([ds[s].to_pandas() for s in ds.keys()], ignore_index=True)
print(f'{len(df):,} tweets')
df['class_label'].value_counts()


## Sampling

Not uniform. Classes describing an emergency in progress are worth more
than sympathy and donation appeals, which are real tweets but not the kind
of message this system carries. Some of the latter are kept anyway — the
encoder must refuse them correctly, and that refusal is part of the test.


## Before sampling: does HumAID contain what this system carries?

A first pass sampled 520 tweets by class and found almost no first-person
distress. 51% were commentary, 18% news reports, 17% retweets, and the
13% that matched first-person patterns were mostly sympathy — *"It breaks
my heart to see my country like this"*.

That is not a flaw in HumAID. It is a corpus of **public tweets about**
disasters. This system carries **messages from people in** them. Annotating
news reports would produce labels for the wrong distribution.

So measure first: how many genuine distress messages exist in all 76,484?
If the answer is a few hundred, they are worth extracting. If it is twenty,
HumAID supplies background text and the real evaluation set has to be
collected.


In [ ]:
import re

# Someone reporting their own emergency: a first-person subject next to a
# need or a predicament. Both halves are required - "we are praying" and
# "people are trapped" (reported) both fail, which is the point.
SUBJECT = re.compile(r"\b(i|we|my|our|us)\b", re.I)
DISTRESS = re.compile(
    r"\b(trapped|stuck|stranded|pinned|buried|can'?t get out|cannot get out|"
    r"need help|help us|rescue us|send help|save us|need water|need food|"
    r"no water|no food|running out|no power|no signal|hurt|injured|bleeding|"
    r"evacuat|shelter|flooded|surrounded)\b", re.I)

# Disqualifiers: reporting about others, or not an emergency at all.
EXCLUDE = re.compile(
    r"(^RT @|\bdeath toll|\bkilled\b|\bkills\b|at least \d+|\bdonat|gofundme|"
    r"fundrais|\bpray|\bthoughts and\b|\bheart ?break|\bso sad\b|\brip\b|"
    r"\breport(s|ed|ing)\b|\bofficials\b|\bauthorities\b|magnitude \d)", re.I)

def is_distress(t: str) -> bool:
    return bool(SUBJECT.search(t) and DISTRESS.search(t) and not EXCLUDE.search(t))

mask = df['tweet_text'].apply(is_distress)
print(f'{mask.sum():,} of {len(df):,} tweets look like first-person distress ({100*mask.mean():.2f}%)\n')

for t in df.loc[mask, 'tweet_text'].sample(min(15, int(mask.sum())), random_state=3):
    print(' *', t[:130])


**Read those fifteen before going further.** If most are genuine calls for
help, re-run the sampling cell below over `df[mask]` instead of `df` and
annotate those. If most are still commentary, tighten the pattern or accept
that the real evaluation set has to be collected from people rather than
found in a corpus.

Either way, say in the paper which one happened. A set drawn from this
filter is *crisis social media*, not *distress messages*, and the difference
is exactly the gap between what was evaluated and what will be deployed.


In [ ]:
# Weight by how likely a class is to contain an actual emergency report.
WEIGHTS = {
    'injured_or_dead_people':              120,
    'requests_or_urgent_needs':            120,
    'rescue_volunteering_or_donation_effort': 60,
    'displaced_people_and_evacuations':     60,
    'infrastructure_and_utility_damage':    60,
    'missing_or_found_people':              40,
    'caution_and_advice':                   20,
    'other_relevant_information':           20,
    'sympathy_and_support':                 10,   # should be SKIPped - tests refusal
    'not_humanitarian':                     10,
    'dont_know_or_cant_judge':               5,
}

rng = np.random.default_rng(42)
parts = []
for cls, n in WEIGHTS.items():
    pool = df[df['class_label'] == cls]
    if len(pool) == 0:
        print(f'  (no rows for {cls})'); continue
    parts.append(pool.sample(min(n, len(pool)), random_state=42))

sample = pd.concat(parts).sample(frac=1, random_state=7).reset_index(drop=True)
sample = sample.drop_duplicates(subset='tweet_text').reset_index(drop=True)
print(f'sampled {len(sample)} tweets')
sample['class_label'].value_counts()


## A suggested category, which is only a suggestion

Four HumAID classes map onto a category cleanly. The rest do not, and the
ones that matter most — TRAPPED, FLOOD, FIRE — have no HumAID class at all.

The suggestion is pre-filled to save typing. **Annotators must read the text
and overwrite it where it is wrong**, which will be often. Accepting the
suggestion unread would just relabel HumAID's taxonomy under new names.


In [ ]:
SUGGEST = {
    'injured_or_dead_people':                 'MEDICAL',
    'requests_or_urgent_needs':               'SUPPLIES_NEEDED',
    'displaced_people_and_evacuations':       'EVACUATION',
    'infrastructure_and_utility_damage':      'COLLAPSE',
    'rescue_volunteering_or_donation_effort': 'RESCUE_NEEDED',
}

out = pd.DataFrame({
    'id':                range(1, len(sample) + 1),
    'tweet_text':        sample['tweet_text'],
    'humaid_label':      sample['class_label'],
    'suggested_category': sample['class_label'].map(SUGGEST).fillna(''),
    'category':          '',
    'casualty_count':    '',
    'mobility':          '',
    'hazard_flags':      '',
    'severity':          '',
    'notes':             '',
})
print(f"{(out['suggested_category'] != '').mean():.0%} of rows got a suggestion; "
      f'the rest start blank')
out.head(8)


## Write the two sheets


In [ ]:
for who in ('A', 'B'):
    out.to_csv(f'annotator_{who}.csv', index=False)

legend = pd.DataFrame({'field': ['category','casualty_count','mobility','hazard_flags','severity'],
 'allowed_values': [
   'TRAPPED COLLAPSE MEDICAL FIRE FLOOD EVACUATION SUPPLIES_NEEDED RESCUE_NEEDED OTHER',
   '0-7  (0 = not stated, 7 = seven or more)',
   'UNKNOWN AMBULATORY TRAPPED UNRESPONSIVE',
   'FIRE WATER GAS STRUCTURAL ELECTRICAL CHEMICAL  (comma-separated, blank = none)',
   'UNKNOWN NONE MINOR SERIOUS']})
legend.to_csv('legend.csv', index=False)

from google.colab import files
for f in ('annotator_A.csv', 'annotator_B.csv', 'legend.csv'):
    files.download(f)
print('downloading 3 files')


## Next

1. Put both CSVs in `data/annotated/` in the repo.
2. Two people label **independently**, following `docs/ANNOTATION_GUIDE.md`.
   Not together — agreement measured after discussion is meaningless.
3. Score it:

   ```bash
   python nlp/score_annotations.py data/annotated/annotator_A.csv data/annotated/annotator_B.csv
   ```

   It prints Cohen's kappa per field, writes the disagreements to work
   through, and then re-runs the encoder comparison on real text.

Budget roughly 6–8 hours per person for ~500 tweets. That is the slow part,
and it is the part that makes every other number in the paper mean something.

**Hindi is not in here.** HumAID is English only, so the code-mixed claim
needs its own collection — scenarios given to 25–30 people, their own words.
Keep it separate and describe it as elicited, not in-the-wild.


---

# A second corpus, closer to what this system carries

HumAID is public tweets *about* disasters. The Figure Eight / Appen
**Disaster Response Messages** set has a `genre` field, and one of its
values is **`direct`** — messages and SMS people sent during the 2010 Haiti
and Chile earthquakes, the 2010 Pakistan floods and Hurricane Sandy,
translated into English.

Those are requests for help, not commentary about a disaster. Its category
labels line up better too: `water`, `food`, `shelter`, `search_and_rescue`,
`medical_help`, `missing_people`, `floods`, `fire`, `earthquake`.

Run this and compare the `direct` samples against the HumAID ones above.
Pick on what you read, not on what either description claims.


In [ ]:
drm = load_dataset('community-datasets/disaster_response_messages',
                   verification_mode='no_checks')
d = pd.concat([drm[s].to_pandas() for s in drm.keys()], ignore_index=True)
print(f'{len(d):,} messages')
print('\ncolumns:', list(d.columns))
if 'genre' in d.columns:
    print('\ngenre:'); print(d['genre'].value_counts())


In [ ]:
# The direct genre is the part worth reading.
direct = d[d['genre'] == 'direct'] if 'genre' in d.columns else d
print(f'{len(direct):,} direct messages\n')
for t in direct['message'].sample(min(20, len(direct)), random_state=11):
    print(' *', str(t)[:130])


### Which corpus to annotate

Compare the twenty above with the HumAID samples. The question is not which
dataset is better in general — HumAID is larger and more carefully
annotated. It is **which one contains the kind of message a person sends
from inside an emergency**, because that is what the encoder will see.

If `direct` reads like genuine requests for help, annotate from there and
keep HumAID as a secondary set, reported separately. Say in the paper that
the choice was made this way: a corpus picked for annotation quality over
domain match is how an evaluation ends up measuring the wrong thing.

Two caveats to check rather than assume:

- These are **translations**. Haitian Creole and Urdu SMS rendered into
  English reads differently from text written in English. Some versions
  carry an `original` column — if it is there, look at it.
- It is from 2010–2012 and smaller than HumAID. Expect a few thousand
  `direct` messages, not tens of thousands.

Still no Hindi in either. That collection has to happen regardless.
